In [ ]:
# ============================================
# SECTION 3: FEATURE ENGINEERING
# ============================================

# Always start with master setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('D:\\Akshaya\\AI training\\python_Projects\\Learningpython1\\Churn_prediction_advance\\data\\raw\\telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].astype(str).str.strip().map({'Yes': 1, 'No': 0}).astype(int)

numeric_cols     = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                    'PhoneService', 'MultipleLines', 'InternetService',
                    'OnlineSecurity', 'TechSupport', 'Contract',
                    'PaperlessBilling', 'PaymentMethod']

print("✅ Data ready!", df.shape)

In [ ]:
# ============================================
# FEATURE ENGINEERING — Creating New Columns
# This is pure business + math thinking
# ============================================

df_feat = df.copy()   # ALWAYS work on a copy, never touch original!

# ── Feature 1: Charges Per Month Ratio ─────────
# TotalCharges should equal tenure × MonthlyCharges
# Difference reveals discounts or price changes
df_feat['ChargesRatio'] = np.where(
    df_feat['tenure'] > 0,
    df_feat['TotalCharges'] / (df_feat['tenure'] * df_feat['MonthlyCharges']),
    0
)

# ── Feature 2: Tenure Groups ───────────────────
# Convert continuous tenure into business segments
df_feat['TenureGroup'] = pd.cut(
    df_feat['tenure'],
    bins   = [0, 12, 24, 48, 72],
    labels = ['New', 'Developing', 'Mature', 'Loyal'],
    right  = True
)

# ── Feature 3: Monthly Charges Group ──────────
df_feat['ChargesGroup'] = pd.cut(
    df_feat['MonthlyCharges'],
    bins   = [0, 35, 65, 95, 120],
    labels = ['Low', 'Medium', 'High', 'Premium'],
    right  = True
)

# ── Feature 4: Number of Services ─────────────
# Count how many services each customer subscribed to
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'TechSupport']

# Convert Yes/No to 1/0 using NumPy — then sum across row
df_feat['NumServices'] = np.sum(
    df_feat[service_cols].apply(
        lambda col: np.where(col == 'Yes', 1, 0)
    ).values,
    axis=1
)

# ── Feature 5: Is AutoPay ──────────────────────
# Binary flag — auto pay customers churn less (we saw this in EDA!)
df_feat['IsAutoPay'] = np.where(
    df_feat['PaymentMethod'].str.contains('automatic', case=False),
    1, 0
)

# ── Feature 6: Is Long Term Contract ──────────
df_feat['IsLongTermContract'] = np.where(
    df_feat['Contract'] == 'Month-to-month', 0, 1
)

# ── Feature 7: High Value Customer ────────────
# Above median monthly charges = high value
median_charge = df_feat['MonthlyCharges'].median()
df_feat['IsHighValue'] = np.where(
    df_feat['MonthlyCharges'] > median_charge, 1, 0
)

# ── Verify new features ────────────────────────
new_features = ['ChargesRatio', 'TenureGroup', 'ChargesGroup',
                'NumServices', 'IsAutoPay', 'IsLongTermContract', 'IsHighValue']

print("✅ New Features Created:")
print(df_feat[new_features].head(10).to_string())
print(f"\nNew shape: {df_feat.shape}")

In [ ]:
# ============================================
# ENCODING — Convert categories to numbers
# ML models only understand numbers!
# ============================================

df_encoded = df_feat.copy()

# ── Drop columns not needed for ML ────────────
df_encoded = df_encoded.drop([
    'customerID',      # just an ID, no predictive value
    'TenureGroup',     # already captured in tenure + NumServices
    'ChargesGroup'     # already captured in MonthlyCharges
], axis=1)

# ── Binary columns — simple map ───────────────
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling']

binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}

for col in binary_cols:
    df_encoded[col] = df_encoded[col].map(binary_map)
    print(f"✅ {col} encoded: {df_encoded[col].unique()}")

# ── MultipleLines — 3 values ──────────────────
df_encoded['MultipleLines'] = df_encoded['MultipleLines'].map(
    {'No phone service': 0, 'No': 1, 'Yes': 2}
)

# ── OnlineSecurity & TechSupport ──────────────
for col in ['OnlineSecurity', 'TechSupport']:
    df_encoded[col] = df_encoded[col].map(
        {'No internet service': 0, 'No': 1, 'Yes': 2}
    )

# ── InternetService — OneHotEncoding ──────────
# 3 categories = no ordinal relationship = use OHE
internet_dummies = pd.get_dummies(
    df_encoded['InternetService'],
    prefix    = 'Internet',
    drop_first = False    # keep all 3 for interpretability
)
df_encoded = pd.concat([df_encoded, internet_dummies], axis=1)
df_encoded = df_encoded.drop('InternetService', axis=1)

# ── Contract — Ordinal Encoding ───────────────
# Has natural order: month < year < two year
df_encoded['Contract'] = df_encoded['Contract'].map(
    {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
)

# ── PaymentMethod — OneHotEncoding ────────────
payment_dummies = pd.get_dummies(
    df_encoded['PaymentMethod'],
    prefix     = 'Payment',
    drop_first = True   # drop first to avoid dummy variable trap
)
df_encoded = pd.concat([df_encoded, payment_dummies], axis=1)
df_encoded = df_encoded.drop('PaymentMethod', axis=1)

print(f"\n✅ Encoding complete!")
print(f"Shape after encoding: {df_encoded.shape}")
print(f"Columns: {df_encoded.columns.tolist()}")

In [ ]:
# ============================================
# SCALING — Normalize numeric features
# Neural Networks especially NEED this!
# ============================================

from sklearn.preprocessing import StandardScaler

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ChargesRatio']

# ── Manual NumPy Scaling first (to understand it) ─
print("📐 MANUAL STANDARDIZATION (NumPy):")
for col in scale_cols:
    data   = df_encoded[col].values
    mean   = np.mean(data)
    std    = np.std(data)
    scaled = (data - mean) / std   # Z-score formula
    print(f"{col}: mean={mean:.2f}, std={std:.2f} → scaled mean={scaled.mean():.4f}")

# ── Professional Sklearn Scaling ──────────────
scaler = StandardScaler()

df_scaled = df_encoded.copy()
df_scaled[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print(f"\n✅ Scaling complete!")
print(df_scaled[scale_cols].describe().round(3).to_string())

In [ ]:
# ============================================
# CLASS IMBALANCE — Critical ML concept!
# Churn = 26% means model can be lazy:
# "predict No Churn every time = 74% accuracy"
# That's USELESS for business!
# ============================================

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# ── Prepare X and y ───────────────────────────
X = df_scaled.drop('Churn', axis=1)
y = df_scaled['Churn']

print("Before SMOTE:")
print(f"Total samples : {len(y)}")
print(f"Churn=0       : {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"Churn=1       : {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

# ── Train Test Split FIRST, then SMOTE ────────
# VERY IMPORTANT: Never apply SMOTE before splitting!
# That causes data leakage!
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y      # keeps churn ratio same in both splits
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set : {X_test.shape}")

# ── Apply SMOTE only on training data ─────────
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE (train only):")
print(f"Churn=0 : {(y_train_balanced==0).sum()}")
print(f"Churn=1 : {(y_train_balanced==1).sum()}")
print("✅ Dataset balanced!")

In [ ]:
# ============================================
# SAVE — So we reuse in Section 4, 5, 6
# ============================================

import os
os.makedirs('../data/processed', exist_ok=True)

# Save as numpy arrays — fast loading for ML
np.save('../data/processed/X_train.npy', X_train_balanced)
np.save('../data/processed/X_test.npy',  X_test)
np.save('../data/processed/y_train.npy', y_train_balanced)
np.save('../data/processed/y_test.npy',  y_test)

# Save column names for reference
pd.Series(X.columns).to_csv('../data/processed/feature_names.csv', index=False)

print("✅ Processed data saved!")
print(f"X_train shape : {X_train_balanced.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Features      : {X.shape[1]}")